# Task 3: Embedding

**Goal:** Embed chunks using OpenAI `text-embedding-3-small`

**Decision:** Local Korean models blocked by macOS 13 dependency conflict (LIM-004)

## 1. Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Verify API key is loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
print("OpenAI API key loaded ✅")

OpenAI API key loaded ✅


In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"Model: text-embedding-3-small")

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: text-embedding-3-small


## 2. Test Single Embedding

In [3]:
# Test with Korean text
test_text = "정수기 필터 교체는 6개월마다 해야 합니다."

embedding = embeddings.embed_query(test_text)

print(f"Text: {test_text}")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")

Text: 정수기 필터 교체는 6개월마다 해야 합니다.
Embedding dimensions: 1536
First 10 values: [-0.0096282958984375, 0.0445556640625, -0.020233154296875, -0.02685546875, -0.03326416015625, 0.010345458984375, -0.007328033447265625, 0.0362548828125, 0.0034503936767578125, -0.00801849365234375]


## 3. Load Chunks from Task 2

In [5]:
import re
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_DIR = Path("../data/raw_pdfs")

def parse_filename(filepath: Path) -> dict:
    """Extract metadata from PDF filename."""
    filename = filepath.stem
    
    category_match = re.match(r'^(waterpurifier|airpurifier|vacuumcleaner)', filename, re.IGNORECASE)
    category = category_match.group(1).lower() if category_match else "unknown"
    
    complexity_match = re.search(r'_(simple|complex)', filename, re.IGNORECASE)
    complexity = complexity_match.group(1).lower() if complexity_match else "unknown"
    
    model_match = re.search(r'([A-Z]{2,3}\d{3,}[A-Z]*)', filename, re.IGNORECASE)
    if model_match:
        model_name = model_match.group(1).upper()
    else:
        model_name = f"{category}_{complexity}"
    
    return {
        "source": filepath.name,
        "category": category,
        "complexity": complexity,
        "model_name": model_name,
    }

def chunk_pdf_with_metadata(filepath: Path, chunk_size: int = 1000, chunk_overlap: int = 200) -> list[dict]:
    """Load PDF and split into chunks with metadata."""
    base_meta = parse_filename(filepath)
    loader = PDFPlumberLoader(str(filepath))
    pages = loader.load()
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    
    all_chunks = []
    for page_doc in pages:
        page_num = page_doc.metadata.get("page", 0) + 1
        page_text = page_doc.page_content
        
        if not page_text.strip():
            continue
        
        page_chunks = splitter.split_text(page_text)
        
        for chunk_idx, chunk_text in enumerate(page_chunks):
            chunk_id = f"{base_meta['model_name']}_p{page_num:03d}_c{chunk_idx:03d}"
            
            chunk_data = {
                "text": chunk_text,
                **base_meta,
                "page": page_num,
                "chunk_id": chunk_id,
                "chunk_index": chunk_idx,
                "char_count": len(chunk_text),
                "section": None,
                "step_number": None,
                "image_ids": [],
            }
            all_chunks.append(chunk_data)
    
    return all_chunks

In [6]:
# Load all chunks
all_chunks = []
for pdf in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_pdf_with_metadata(pdf)
    all_chunks.extend(chunks)
    print(f"{pdf.name}: {len(chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

airpurifier_complex_MFL69726859_00_190321_00.pdf: 67 chunks
airpurifier_simple.pdf: 58 chunks
vaccumcleaner_complex.pdf: 69 chunks
vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf: 26 chunks
waterpurifier_complex.pdf: 56 chunks
waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf: 44 chunks

Total: 320 chunks


## 4. Embed All Chunks

In [7]:
# Extract texts for embedding
texts = [chunk["text"] for chunk in all_chunks]

print(f"Embedding {len(texts)} chunks...")
print(f"Estimated cost: ~${len(texts) * 0.00002:.4f} (text-embedding-3-small)")

Embedding 320 chunks...
Estimated cost: ~$0.0064 (text-embedding-3-small)


In [8]:
# Embed all texts (batch)
import time

start = time.time()
all_embeddings = embeddings.embed_documents(texts)
elapsed = time.time() - start

print(f"Embedded {len(all_embeddings)} chunks in {elapsed:.2f}s")
print(f"Embedding dimensions: {len(all_embeddings[0])}")

Embedded 320 chunks in 2.98s
Embedding dimensions: 1536


## 5. Verify Embeddings

In [9]:
# Basic sanity checks
import numpy as np

embeddings_array = np.array(all_embeddings)

print(f"Shape: {embeddings_array.shape}")
print(f"Dtype: {embeddings_array.dtype}")
print(f"\nStats:")
print(f"  Min: {embeddings_array.min():.4f}")
print(f"  Max: {embeddings_array.max():.4f}")
print(f"  Mean: {embeddings_array.mean():.4f}")
print(f"  Std: {embeddings_array.std():.4f}")

Shape: (320, 1536)
Dtype: float64

Stats:
  Min: -0.1495
  Max: 0.1614
  Mean: -0.0009
  Std: 0.0255


In [10]:
# Test similarity between related chunks
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Compare first chunk of each PDF
print("Cosine similarity between first chunks of different PDFs:")
print("-" * 50)

# Find first chunk of each category
categories = {}
for i, chunk in enumerate(all_chunks):
    key = f"{chunk['category']}_{chunk['complexity']}"
    if key not in categories:
        categories[key] = i

keys = list(categories.keys())
for i, k1 in enumerate(keys):
    for k2 in keys[i+1:]:
        idx1, idx2 = categories[k1], categories[k2]
        sim = cosine_similarity(all_embeddings[idx1], all_embeddings[idx2])
        print(f"{k1} <-> {k2}: {sim:.4f}")

Cosine similarity between first chunks of different PDFs:
--------------------------------------------------
airpurifier_complex <-> airpurifier_simple: 0.9726
airpurifier_complex <-> unknown_complex: 0.6929
airpurifier_complex <-> unknown_simple: 0.7659
airpurifier_complex <-> waterpurifier_complex: 0.7642
airpurifier_complex <-> waterpurifier_simple: 0.7558
airpurifier_simple <-> unknown_complex: 0.6979
airpurifier_simple <-> unknown_simple: 0.7656
airpurifier_simple <-> waterpurifier_complex: 0.7746
airpurifier_simple <-> waterpurifier_simple: 0.7700
unknown_complex <-> unknown_simple: 0.7916
unknown_complex <-> waterpurifier_complex: 0.7088
unknown_complex <-> waterpurifier_simple: 0.7032
unknown_simple <-> waterpurifier_complex: 0.7410
unknown_simple <-> waterpurifier_simple: 0.7443
waterpurifier_complex <-> waterpurifier_simple: 0.9691


## 6. Notes

### Results
- (fill after running)

### Observations
- (fill after running)